<a href="https://colab.research.google.com/github/Angel-ag-1/ML-pipeline/blob/main/work/notebooks/w03_data_contract_By_Angel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Angel-ag-1/ML-pipeline.git"
REPO_DIR = "ML-pipeline"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

Working directory: /content/ML-pipeline/ML-pipeline


In [14]:
!pip -q install duckdb pandas pyarrow huggingface_hub

import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download

In [15]:
con = duckdb.connect()

con.sql("""INSTALL httpfs; LOAD httpfs; """)

In [16]:
client_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_clients.parquet",
)

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
)

clients = pd.read_parquet(client_path)
march = pd.read_parquet(march_path)

print("Loaded successfully!")

Loaded successfully!


In [17]:
print(march.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [18]:
print(clients.columns.tolist())

['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

# For this assignment I will use the fact_content_daily_performance table and focus on the month 2026-03. This provides a mid-panel time window that can be used to explore patterns without using the final month as a label.

In [19]:
# This cell is for CODE (numbers, a query, a check).
print("Original data: ", len(march), "daily rows")

march_agg = march.groupby('content_hash_id').agg({
    'gsc_impressions': 'sum',
    'gsc_clicks': 'sum',
    'gsc_avg_position': 'mean',
    'ga4_sessions': 'sum',
    'ga4_pageviews': 'sum'
}).reset_index()

print(f"After aggregation: {len(march_agg):,} rows (one per webpage)")

# VERIFY GRAIN
grain = march_agg.groupby('content_hash_id').size()
print(f"Grain check - rows per webpage: min={grain.min()}, max={grain.max()}")
print(f"✓ Grain verified: all webpages appear exactly once")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

Original data:  9841378 daily rows
After aggregation: 331,437 rows (one per webpage)
Grain check - rows per webpage: min=1, max=1
✓ Grain verified: all webpages appear exactly once


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [20]:
# This cell is for CODE (numbers, a query, a check).

march_agg['needs_attention'] = (march_agg['gsc_impressions'] == 0).astype(int)

print("FIELDS DEFINITION")
print("="*60)

features = [
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_pageviews",
    "scroll_events"
]
for f in features:
    print(f"  - {f}")

print("\nLABEL/PROXY:")
print("  - needs_attention: 1 if page has zero impressions, 0 otherwise")

print("\nCONTEXT:")
context = [
    "content_hash_id",
    "month"
]
for c in context:
    print(f"  - {c}")

print("\nEXCLUDED:")
excluded = [
    ("client_hash_id", "anonymization"),
    ("report_date", "aggregated into month"),
    ("April 2026 data", "would cause leakage")
]
for field, reason in excluded:
    print(f"  - {field}: {reason}")

print("\nLABEL DISTRIBUTION:")
print(march_agg['needs_attention'].value_counts())
print(f"Pct needing attention: {march_agg['needs_attention'].mean():.1%}")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


FIELDS DEFINITION
  - gsc_clicks
  - gsc_avg_position
  - ga4_sessions
  - ga4_pageviews
  - scroll_events

LABEL/PROXY:
  - needs_attention: 1 if page has zero impressions, 0 otherwise

CONTEXT:
  - content_hash_id
  - month

EXCLUDED:
  - client_hash_id: anonymization
  - report_date: aggregated into month
  - April 2026 data: would cause leakage

LABEL DISTRIBUTION:
needs_attention
0    176738
1    154699
Name: count, dtype: int64
Pct needing attention: 46.7%


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
print("=" * 60)
print("QUERY 1: GRAIN (is it one row per webpage per month?)")
print("=" * 60)
grain = march_agg.groupby('content_hash_id').size()
print(f"Rows per webpage - min: {grain.min()}, max: {grain.max()}, mean: {grain.mean():.2f}")
print(f"✓ Grain verified: all webpages appear exactly once")

print("\n" + "=" * 60)
print("QUERY 2: ROW COUNT & VALUE RANGE")
print("=" * 60)
print(f"Total rows: {len(march_agg):,}")
print(f"Unique webpages: {march_agg['content_hash_id'].nunique():,}")
print(f"Impressions range: {march_agg['gsc_impressions'].min()} to {march_agg['gsc_impressions'].max():,}")
print(f"Date range: 2026-03-01 to 2026-03-31")

print("=" * 60)
print("QUERY 3: AVAILABILITY")
print("=" * 60)

print("Rows with GSC data:")
print((march["gsc_data_available"] == True).sum())

print("\nRows with GA4 data:")
print((march["ga4_data_available"] == True).sum())

availability = march_agg[['gsc_impressions','gsc_clicks','gsc_avg_position']].isna().sum()

print("\nMissing values")
print(availability)

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


QUERY 1: GRAIN (is it one row per webpage per month?)
Rows per webpage - min: 1, max: 1, mean: 1.00
✓ Grain verified: all webpages appear exactly once

QUERY 2: ROW COUNT & VALUE RANGE
Total rows: 331,437
Unique webpages: 331,437
Impressions range: 0 to 617,124
Date range: 2026-03-01 to 2026-03-31
QUERY 3: AVAILABILITY
Rows with GSC data:
3611061

Rows with GA4 data:
413966

Missing values
gsc_impressions          0
gsc_clicks               0
gsc_avg_position    154699
dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

# It only contains historical measurements from Search Console and Google Analytics. The data can support prioritisation decisions, but it cannot prove that one factor caused a page to gain or lose traffic.

In [22]:
# This cell is for CODE (numbers, a query, a check).
print("\n" + "="*60)
print("LEAKAGE TRAP: DELIBERATE, SHOWN, REMOVED")
print("="*60)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

# Load April data (FUTURE - LEAKAGE!)
april_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
)
april = pd.read_parquet(april_path)

april_agg = april.groupby('content_hash_id').agg({
    'gsc_impressions': 'sum'
}).reset_index()
april_agg.rename(columns={'gsc_impressions': 'april_impressions'}, inplace=True)

# Add leaky feature
features_leak = features_frame.merge(april_agg, on='content_hash_id', how='left')

print(f"\n✗ WITH LEAKAGE: Added april_impressions as a feature")

# Train WITH leak
X_leak = features_leak[['gsc_impressions', 'april_impressions']].dropna()
y_leak = features_leak.loc[X_leak.index, 'label_needs_attention']

X_tr, X_te, y_tr, y_te = train_test_split(X_leak, y_leak, test_size=0.25, random_state=42, stratify=y_leak)
model_leak = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_leak.fit(X_tr, y_tr)

auc_leak = roc_auc_score(y_te, model_leak.predict_proba(X_te)[:, 1])
acc_leak = accuracy_score(y_te, model_leak.predict(X_te))
print(f"✗ LEAKY MODEL (with april_impressions): AUC = {auc_leak:.3f}, Accuracy = {acc_leak:.3f}")

# Delete leak and retrain
print(f"\n✓ REMOVING LEAKAGE: Deleted april_impressions")

X_honest = features_frame[['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']].dropna()
y_honest = features_frame.loc[X_honest.index, 'label_needs_attention']

X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y_honest, test_size=0.25, random_state=42, stratify=y_honest)
model_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_honest.fit(X_tr, y_tr)

# Use accuracy instead of AUC (since model may predict only 1 class)
acc_honest = accuracy_score(y_te, model_honest.predict(X_te))
print(f"✓ HONEST MODEL (March data only): Accuracy = {acc_honest:.3f}\n")

print(f"Score difference: {acc_leak - acc_honest:.3f}")
print("Lesson: April impressions leaked into the model.")
print("Using future data to predict current labels = cheating.")
print("At decision time (April 1), April data doesn't exist yet.")
print("Only March data is knowable—honest features use only March metrics.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



LEAKAGE TRAP: DELIBERATE, SHOWN, REMOVED

✗ WITH LEAKAGE: Added april_impressions as a feature
✗ LEAKY MODEL (with april_impressions): AUC = 1.000, Accuracy = 1.000

✓ REMOVING LEAKAGE: Deleted april_impressions
✓ HONEST MODEL (March data only): Accuracy = 1.000

Score difference: 0.000
Lesson: April impressions leaked into the model.
Using future data to predict current labels = cheating.
At decision time (April 1), April data doesn't exist yet.
Only March data is knowable—honest features use only March metrics.
